# Feature Engineering (Missing Value Imputation)
----
- Iterative Imputer
- MICE- Multiple Imputation by Chained Equations
- Missing Completely at Random (MCAR)
- Missing at Random (MAR)
- Missing Not at Random (MNAR)
- Find Predictive Value for Iterative Imputer Technique


# Import Libraries

In [130]:
import pandas as pd 
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

# Import Dataset

In [131]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


# Remove Target Column

In [132]:
df = df.iloc[:,0:-1] 
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


# NaN value import (Manipulate the Data)

In [154]:
df.iloc[1, 0] = np.NaN
df.iloc[3, 1] = np.NaN
df.iloc[-1, -1] = np.NaN

In [155]:
df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


# Step 1 - Impute all missing values with mean

In [135]:
df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean()) 
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean()) 
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [136]:
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


# Step 2 - Remove the column 1 imputed value (Left to Right)

In [139]:
df1 = df0.copy() 
df1.iloc[1,0] = np.NaN 
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


# Training Data in X (Training Input)

In [140]:
X = df1.iloc[[0,2,3,4],1:3] 
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


# Training Data in Y (Corresponding Output)

In [141]:
y = df1.iloc[[0,2,3,4],0] 
y

21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

# Step 3 - Predict missing value of column 1

In [142]:
lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

/Users/arjan/env/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [34]:
df1.iloc[1,0] = 23.14

In [143]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


# Step 4 - Remove the column 2 imputed value (Left to Right)

In [144]:
df1.iloc[3,1] = np.NaN 
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.00
37,NaN,5.0,20.00
2,15.0,10.0,41.00
14,12.0,NaN,26.00
44,2.0,15.0,29.25


# Training Data in X (Training Input) | Column 2

In [145]:
X = df1.iloc[[0,1,2,4],[0,2]] 
X

,R&D Spend,Marketing Spend
21,8.0,30.00
37,NaN,20.00
2,15.0,41.00
44,2.0,29.25


# Training Data in Y (Corresponding Output) | Column 2

In [146]:
y = df1.iloc[[0,1,2,4],1] 
y

21    15.0
37     5.0
2     10.0
44    15.0
Name: Administration, dtype: float64

# Step 5 - Predict missing value of column 2

In [156]:
lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [148]:
df1.iloc[3,1] = 11.06

In [149]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,29.25


# Step 6 - Remove the column 3 imputed value (Left to Right)

In [150]:
df1.iloc[4,-1] = np.NaN 
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.0
37,NaN,5.00,20.0
2,15.0,10.00,41.0
14,12.0,11.06,26.0
44,2.0,15.00,NaN


# Training Data in X (Training Input) | Column 3

In [151]:
X = df1.iloc[0:4,0:2] 
X

,R&D Spend,Administration
21,8.0,15.00
37,NaN,5.00
2,15.0,10.00
14,12.0,11.06


# Training Data in Y (Corresponding Output) | Column 3

In [152]:
y = df1.iloc[0:4,-1] 
y

21    30.0
37    20.0
2     41.0
14    26.0
Name: Marketing Spend, dtype: float64

# Step 7 - Predict missing value of column 3

In [153]:
lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [47]:
df1.iloc[4,-1] = 31.56

In [48]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


# Step 8 - Subtract 0th (df0) iteration from 1st (df1) iteration

In [49]:
df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


# Again Iteration Process

In [50]:
df2 = df1.copy() 
df2.iloc[1,0] = np.NaN 
df2

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,31.56


In [52]:
X = df2.iloc[[0,2,3,4],1:3] 
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

/Users/arjan/env/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.78627207])

In [53]:
df2.iloc[1,0] = 23.78

In [54]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [55]:
df2.iloc[3,1] = np.NaN
X = df2.iloc[[0,1,2,4],[0,2]] 
y= df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

/Users/arjan/env/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.22020174])

In [56]:
df2.iloc[3,1] = 11.22

In [57]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [58]:
df2.iloc[4,-1] = np.NaN 

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

/Users/arjan/env/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([38.87979054])

In [60]:
df2.iloc[4,-1] = 31.56

In [61]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [62]:
df2 - df1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.0
37,0.64,0.00,0.0
2,0.00,0.00,0.0
14,0.00,0.16,0.0
44,0.00,0.00,0.0


In [63]:
df3 = df2.copy() 
df3.iloc[1,0] = np.NaN 
df3

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.22,26.00
44,2.0,15.00,31.56


In [64]:
X = df3.iloc[[0,2,3,4],1:3] 
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

/Users/arjan/env/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([24.57698058])

In [67]:
df3.iloc[1,0] = 24.57

In [68]:
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,24.57,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [70]:
df3.iloc[3,1] = np.NaN

X = df3.iloc[[0,1,2,4],[0,2]] 
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

/Users/arjan/env/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.37282844])

In [71]:
df3.iloc[3,1] = 11.37

In [72]:
df3.iloc[4,-1] = np.NaN 

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y) 
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

/Users/arjan/env/lib/python3.8/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([45.53976417])

In [73]:
df3.iloc[4,-1] = 45.53

In [74]:
df2.iloc[3,1] = 11.22

In [75]:
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,24.57,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.37,26.00
44,2.00,15.00,45.53


In [77]:
df3 - df2

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.79,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.15,0.00
44,0.00,0.00,13.97
